# 🍷 Lab 7: Wine Quality Prediction

**Challenge**: Predict wine quality (good/bad) from chemical measurements.

**What you'll use**: pandas, scikit-learn, Gradient Boosting, feature engineering

---
### The Story
A winery wants to automate quality control. Instead of having a human taste every bottle, they measure 11 chemical properties (acidity, sugar, alcohol, etc.) and want to predict if the wine is good quality or not. This is a real Kaggle dataset!

In [ ]:
!pip install pandas scikit-learn matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

print('✅ Ready!')

In [ ]:
# Load the Wine Quality dataset directly from UCI
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
df = pd.read_csv(url, sep=';')

print('Shape:', df.shape)
print('\nQuality distribution:')
print(df['quality'].value_counts().sort_index())
df.head()

In [ ]:
# Convert to binary: quality >= 7 is "good" (1), else "bad" (0)
df['good_wine'] = (df['quality'] >= 7).astype(int)

print('Good wines:', df['good_wine'].sum())
print('Bad wines:', (df['good_wine'] == 0).sum())
print(f'\nClass balance: {df["good_wine"].mean():.1%} good wines')

# Visualize quality distribution
df['quality'].value_counts().sort_index().plot(kind='bar', color='#ff6b6b', figsize=(8,4))
plt.title('Wine Quality Distribution')
plt.xlabel('Quality Score')
plt.ylabel('Count')
plt.axvline(x=3.5, color='green', linestyle='--', label='Good threshold (≥7)')
plt.legend()
plt.show()

In [ ]:
# Explore: What makes a good wine?
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
features_to_plot = ['alcohol', 'volatile acidity', 'sulphates', 'citric acid', 'density', 'pH']

for ax, feature in zip(axes.flat, features_to_plot):
    df.boxplot(column=feature, by='good_wine', ax=ax)
    ax.set_title(feature)
    ax.set_xlabel('0=Bad, 1=Good')

plt.suptitle('Feature Distributions: Good vs Bad Wine')
plt.tight_layout()
plt.show()
print('Notice: Higher alcohol and sulphates → better wine!')

In [ ]:
# Prepare features
feature_cols = [c for c in df.columns if c not in ['quality', 'good_wine']]
X = df[feature_cols]
y = df['good_wine']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')
print(f'Features: {list(feature_cols)}')

In [ ]:
# YOUR TURN: Train a Gradient Boosting Classifier
# This is one of the most powerful algorithms for tabular data!
# Hint: GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)

gb_model = ___  # ← Create GradientBoostingClassifier
___  # ← Fit on training data

gb_preds = gb_model.predict(X_test)
gb_acc = accuracy_score(y_test, gb_preds)
gb_auc = roc_auc_score(y_test, gb_model.predict_proba(X_test)[:, 1])

print(f'Gradient Boosting Accuracy: {gb_acc:.1%}')
print(f'Gradient Boosting AUC: {gb_auc:.3f}')

In [ ]:
# Feature importance — what makes wine good?
importance = pd.Series(gb_model.feature_importances_, index=feature_cols).sort_values(ascending=True)

importance.plot(kind='barh', color='#00ff9d', figsize=(8, 6))
plt.title('What Predicts Wine Quality?')
plt.xlabel('Feature Importance')
plt.tight_layout()
plt.show()

print(f'Most important feature: {importance.index[-1]}')
print(f'Least important feature: {importance.index[0]}')

In [ ]:
# YOUR TURN: Use cross-validation for a more reliable accuracy estimate
# Cross-validation tests the model on 5 different splits of the data
# Hint: cross_val_score(gb_model, X, y, cv=5, scoring='accuracy')

cv_scores = ___  # ← Run 5-fold cross validation
cv_mean = cv_scores.mean()

print(f'Cross-validation scores: {[f"{s:.1%}" for s in cv_scores]}')
print(f'Mean CV accuracy: {cv_mean:.1%} ± {cv_scores.std():.1%}')

In [ ]:
# ✅ TEST CELL
assert hasattr(gb_model, 'predict'), 'gb_model must be trained'
assert gb_acc >= 0.82, f'Accuracy should be >= 82%, got {gb_acc:.1%}'
assert gb_auc >= 0.80, f'AUC should be >= 0.80, got {gb_auc:.3f}'
assert len(cv_scores) == 5, 'cv_scores should have 5 values (5-fold CV)'
print('🎉 All tests passed!')
print(f'Accuracy: {gb_acc:.1%} | AUC: {gb_auc:.3f} | CV Mean: {cv_mean:.1%}')
print(f'Most important predictor: {importance.index[-1]}')